# README Benchmark Runner

Run the cells top to bottom. Edit only the configuration cell for normal experiment changes.

## Configuration

In [ ]:
EXPERIMENT = {
    "run_name": "default_run",
    "data_csv": "data/repo_list.csv",
    "tools": {
        "osa": {"enabled": True},
        "readmeready": {"enabled": True},
        "larch": {"enabled": True, "mode": "local"},
    },
    "models": [
        "openai/gpt-4.1",
        "anthropic/claude-sonnet-4",
        "google/gemma-3-27b-it",
    ],
    "judge": {
        "api": "openrouter",
        "base_url": "https://openrouter.ai/api/v1",
        "model": "gpt-4.1",
    },
    "reset": {
        "repos": False,
        "tool_outputs": True,
        "evaluation": False,
    },
    # Set to an integer for smoke tests, or None for the full CSV.
    "limit_repos": None,
}


## Imports And Paths

In [2]:
from pathlib import Path
import importlib
import sys

cwd = Path.cwd()
candidates = [cwd, cwd / "benchmark_README", *cwd.parents]
BM = next((p for p in candidates if (p / "src" / "notebook_utils.py").is_file()), cwd)
if str(BM) not in sys.path:
    sys.path.insert(0, str(BM))

import src.notebook_utils as notebook_utils
import src.tool_runners as tool_runners
import src.evaluation as evaluation

importlib.reload(notebook_utils)
importlib.reload(tool_runners)
importlib.reload(evaluation)

from src.notebook_utils import build_paths, ensure_directories, load_env_file, read_repo_table, prepare_repositories
from src.tool_runners import run_selected_tools
from src.evaluation import evaluate_outputs

load_env_file(BM / ".env")
paths = build_paths(EXPERIMENT["run_name"])
ensure_directories(paths)
print(paths)


BenchmarkPaths(benchmark_root=WindowsPath('D:/VKR_README_EVAL/benchmark_README'), workspace_root=WindowsPath('D:/VKR_README_EVAL'), experiment_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run'), repositories_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run/repositories'), original_readmes_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run/original_readmes'), structures_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run/repo_structures'), logs_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run/logs'), tools_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run/tools'), evaluation_dir=WindowsPath('D:/VKR_README_EVAL/benchmark_README/results/default_run/evaluation'))


## Load Repository Table

In [3]:
repo_df = read_repo_table(EXPERIMENT["data_csv"])
repo_df = repo_df.iloc[[0]]
if EXPERIMENT.get("limit_repos"):
    repo_df = repo_df.head(int(EXPERIMENT["limit_repos"]))
repo_df.head()


,repo_url,repo_name,commit_sha,commit_date,repository,repo_slug
0,https://github.com/AntonOsika/gpt-engineer,AntonOsika/gpt-engineer,a90fcd543eedcc0ff2c34561bc0785d2ba83c47e,2024-11-17T22:42:12+00:00,https://github.com/AntonOsika/gpt-engineer,gpt-engineer


## Pre-flight: Clone, Checkout, Snapshot

In [4]:
preflight = prepare_repositories(
    repo_df,
    paths,
    reset=bool(EXPERIMENT.get("reset", {}).get("repos", False)),
)
preflight


,repo_url,repo_name,repo_slug,commit_sha,status,started_at,finished_at,repo_dir,original_readme,structure_json,error
0,https://github.com/AntonOsika/gpt-engineer,AntonOsika/gpt-engineer,gpt-engineer,a90fcd543eedcc0ff2c34561bc0785d2ba83c47e,failed,2026-05-18T00:25:54+00:00,2026-05-18T00:25:54+00:00,,,,[WinError 5] Отказано в доступе: 'D:\\VKR_READ...


## Run Selected Tools

In [5]:
tool_status = run_selected_tools(repo_df, paths, EXPERIMENT)
tool_status.tail(20)


,tool,model,model_label,repo_slug,status,started_at,finished_at,duration_sec,output_path,log_path,error
1,osa,openai/gpt-4.1,gpt-4.1,gpt-engineer,done,2026-05-18T00:25:56+00:00,2026-05-18T00:27:02+00:00,66.325,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,NaN
2,readmeready,openai/gpt-4.1,gpt-4.1,gpt-engineer,done,2026-05-18T00:27:02+00:00,2026-05-18T00:30:35+00:00,212.941,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,NaN
3,larch,openai/gpt-4.1,gpt-4.1,gpt-engineer,done,2026-05-18T00:30:35+00:00,2026-05-18T00:31:00+00:00,24.309,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,NaN
4,osa,anthropic/claude-sonnet-4,claude-sonnet-4,gpt-engineer,done,2026-05-18T00:31:00+00:00,2026-05-18T00:32:27+00:00,87.593,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,NaN
5,readmeready,anthropic/claude-sonnet-4,claude-sonnet-4,gpt-engineer,done,2026-05-18T00:32:27+00:00,2026-05-18T00:36:42+00:00,254.931,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,NaN
6,larch,anthropic/claude-sonnet-4,claude-sonnet-4,gpt-engineer,done,2026-05-18T00:36:42+00:00,2026-05-18T00:37:48+00:00,65.579,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,NaN
7,osa,google/gemma-3-27b-it,gemma-3-27b-it,gpt-engineer,done,2026-05-18T00:37:48+00:00,2026-05-18T00:39:21+00:00,92.902,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,NaN
8,readmeready,google/gemma-3-27b-it,gemma-3-27b-it,gpt-engineer,done,2026-05-18T00:39:21+00:00,2026-05-18T00:43:22+00:00,241.354,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,NaN
9,larch,google/gemma-3-27b-it,gemma-3-27b-it,gpt-engineer,done,2026-05-18T00:43:22+00:00,2026-05-18T00:43:55+00:00,33.016,D:\VKR_README_EVAL\benchmark_README\results\de...,D:\VKR_README_EVAL\benchmark_README\results\de...,


## Evaluate Generated READMEs

In [6]:
eval_rows = evaluate_outputs(
    paths,
    EXPERIMENT,
    reset=bool(EXPERIMENT.get("reset", {}).get("evaluation", False)),
)
eval_rows.tail(20)


d:\VKR_README_EVAL\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

,tool,model,model_label,repo_slug,score,reason,success,output_path
0,osa,openai/gpt-4.1,gpt-4.1,gpt-engineer,1.0,The README in the actual output fully aligns w...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
1,readmeready,openai/gpt-4.1,gpt-4.1,gpt-engineer,0.8,"The actual output README is comprehensive, wel...",True,D:\VKR_README_EVAL\benchmark_README\results\de...
2,larch,openai/gpt-4.1,gpt-4.1,gpt-engineer,0.9,"The provided README is comprehensive, addressi...",True,D:\VKR_README_EVAL\benchmark_README\results\de...
3,osa,anthropic/claude-sonnet-4,claude-sonnet-4,gpt-engineer,1.0,The README.md is present in the repository str...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
4,readmeready,anthropic/claude-sonnet-4,claude-sonnet-4,gpt-engineer,0.7,The actual output README covers the repository...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
5,larch,anthropic/claude-sonnet-4,claude-sonnet-4,gpt-engineer,0.9,The README in the actual output strongly align...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
6,osa,google/gemma-3-27b-it,gemma-3-27b-it,gpt-engineer,1.0,The README in the actual output fully aligns w...,True,D:\VKR_README_EVAL\benchmark_README\results\de...
7,readmeready,google/gemma-3-27b-it,gemma-3-27b-it,gpt-engineer,0.7,"The actual output README is comprehensive, wel...",True,D:\VKR_README_EVAL\benchmark_README\results\de...
8,larch,google/gemma-3-27b-it,gemma-3-27b-it,gpt-engineer,0.8,"The actual output README is comprehensive, wel...",True,D:\VKR_README_EVAL\benchmark_README\results\de...


## Summary

In [ ]:
summary_path = paths.evaluation_dir / "final_summary.csv"
if summary_path.is_file():
    import pandas as pd
    display(pd.read_csv(summary_path))
else:
    print("No summary yet. Run evaluation after at least one successful tool output.")
